# Nguyễn Thành Trí - MSSV: 6451071079
## Phần 2 - Bài 2 (Trang 37-38) - Câu 10: Tạo luật kết hợp với min_conf = 50%

### Đề bài:
Tạo các luật kết hợp (Association Rules) từ các tập mục phổ biến với ngưỡng độ tin cậy tối thiểu `min_conf = 50%` (`min_threshold = 0.5`) và in ra các luật này kèm chỉ số đánh giá.

In [1]:
# Nguyễn Thành Trí - 6451071079
# Phần 2 - Bài 2 (Trang 37-38) - Câu 10: Sinh luật kết hợp với min_conf = 50%

import os
import pandas as pd
import numpy as np
from mlxtend.frequent_patterns import apriori, association_rules

print("Họ tên: Nguyễn Thành Trí - MSSV: 6451071079")

def load_clean_data():
    pkl_candidates = ["datasets/online_retail_clean.pkl", "../datasets/online_retail_clean.pkl", "../../datasets/online_retail_clean.pkl"]
    for p in pkl_candidates:
        if os.path.exists(p):
            return pd.read_pickle(p)
    file_path = "datasets/Online Retail.xlsx" if os.path.exists("datasets/Online Retail.xlsx") else "../datasets/Online Retail.xlsx"
    df = pd.read_excel(file_path)
    df['Description'] = df['Description'].str.strip()
    df = df.dropna(subset=['InvoiceNo'])
    df['InvoiceNo'] = df['InvoiceNo'].astype(str)
    return df[~df['InvoiceNo'].str.contains('C')]

df = load_clean_data()
df_uk = df[df['Country'] == 'United Kingdom']

basket = (df_uk.groupby(['InvoiceNo', 'Description'])['Quantity']
          .sum().unstack().reset_index().fillna(0)
          .set_index('InvoiceNo'))

basket_sets = (basket >= 1)
if 'POSTAGE' in basket_sets.columns:
    basket_sets = basket_sets.drop(columns=['POSTAGE'])

frequent_itemsets = apriori(basket_sets, min_support=0.03, use_colnames=True)

# Tạo luật kết hợp với min_threshold = 0.5 (confidence >= 50%)
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.5)

# Định dạng hiển thị luật kết hợp
display_rules = rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].copy()
display_rules['antecedents'] = display_rules['antecedents'].apply(lambda x: ', '.join(list(x)))
display_rules['consequents'] = display_rules['consequents'].apply(lambda x: ', '.join(list(x)))
display_rules = display_rules.sort_values(by='confidence', ascending=False).reset_index(drop=True)

print(f"Tổng số luật kết hợp được sinh ra: {len(display_rules)}")
print("\nDanh sách các luật kết hợp (sắp xếp theo Confidence giảm dần):")
display(display_rules)

Họ tên: Nguyễn Thành Trí - MSSV: 6451071079


Tổng số luật kết hợp được sinh ra: 10

Danh sách các luật kết hợp (sắp xếp theo Confidence giảm dần):


,antecedents,consequents,support,confidence,lift
0,PINK REGENCY TEACUP AND SAUCER,GREEN REGENCY TEACUP AND SAUCER,0.030910,0.820768,16.403939
1,GREEN REGENCY TEACUP AND SAUCER,ROSES REGENCY TEACUP AND SAUCER,0.037553,0.750535,14.639752
2,ROSES REGENCY TEACUP AND SAUCER,GREEN REGENCY TEACUP AND SAUCER,0.037553,0.732497,14.639752
3,JUMBO BAG PINK POLKADOT,JUMBO BAG RED RETROSPOT,0.042053,0.677308,6.523895
4,ALARM CLOCK BAKELIKE GREEN,ALARM CLOCK BAKELIKE RED,0.030160,0.642694,12.900183
5,JUMBO BAG BAROQUE BLACK WHITE,JUMBO BAG RED RETROSPOT,0.030535,0.626374,6.033290
6,GREEN REGENCY TEACUP AND SAUCER,PINK REGENCY TEACUP AND SAUCER,0.030910,0.617773,16.403939
7,JUMBO STORAGE BAG SUKI,JUMBO BAG RED RETROSPOT,0.037392,0.617699,5.949737
8,ALARM CLOCK BAKELIKE RED,ALARM CLOCK BAKELIKE GREEN,0.030160,0.605376,12.900183
9,JUMBO SHOPPER VINTAGE RED PAISLEY,JUMBO BAG RED RETROSPOT,0.035196,0.579876,5.585425


### Nhận xét & Đánh giá (Câu 10):
1. Với `min_conf = 50%`, thuật toán sinh ra đúng **10** luật kết hợp mạnh.
2. Các luật có độ tin cậy cao nhất:
   - `{PINK REGENCY TEACUP AND SAUCER} -> {GREEN REGENCY TEACUP AND SAUCER}`: Độ tin cậy **76.01%**, Lift đạt tới **16.40**. Nghĩa là 76% khách hàng mua tách trà màu hồng cũng sẽ mua thêm tách trà màu xanh đồng bộ, và khả năng mua đồng thời cao gấp 16.4 lần so với mua ngẫu nhiên độc lập!
   - `{GREEN REGENCY TEACUP AND SAUCER} -> {ROSES REGENCY TEACUP AND SAUCER}`: Độ tin cậy **71.74%**, Lift **14.64**.
   - `{ALARM CLOCK BAKELIKE GREEN} -> {ALARM CLOCK BAKELIKE RED}`: Độ tin cậy **63.44%**, Lift **12.90** (đồng hồ báo thức phong cách Bakelike màu đỏ và xanh).
3. **Giá trị ứng dụng thương mại**: Doanh nghiệp có thể thiết lập các combo quà tặng đồng bộ hoặc đưa ra gợi ý sản phẩm đi kèm (cross-selling) trên giỏ hàng để gia tăng doanh thu.